In [ ]:
from transformers import AutoTokenizer

tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
tokenizer_base.save_pretrained("Qwen2.5-7B-BASE")

In [ ]:
def checkList(list_tokens, vocab_to_check):
    # print('++Checking list of tokens:', list_tokens)
    merged_tokens = list_tokens[:]
    longest_token = ""
    start_idx, end_idx = -1, -1
    
    for i in range(len(list_tokens)):
        for j in range(i + 1, len(list_tokens) + 1):
            candidate = "".join(list_tokens[i:j])
            if candidate in vocab_to_check:
                if len(candidate) > len(longest_token):
                    longest_token = candidate
                    start_idx, end_idx = i, j

    if longest_token:
        merged_tokens = list_tokens[:start_idx] + [longest_token] + list_tokens[end_idx:]

    # print('--Merged tokens:', merged_tokens)
    return merged_tokens

In [ ]:
import os, glob
import pickle as pkl
import json
from collections import defaultdict
os.makedirs("./VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/",exist_ok=True)

for fname in glob.glob("./VocabFiles/CPT-SupremeCourtCases/CPT-SupremeCourtCases-Qwen2.5_Vocab/*.txt"):
    
    print('***********Processing:',fname)
    # continue
    
    vocab_to_merges = defaultdict(list)
    
    tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
    tokenizer_json = json.load(open("./Qwen2.5-7B-BASE/tokenizer.json", 'r', encoding='utf-8'))
    vocab_base = tokenizer_json["model"]["vocab"]
    merges_base = tokenizer_json["model"]["merges"]
    
    words_to_add = open(fname,'r',encoding='utf-8').read().splitlines()
    words_to_add = sorted(words_to_add, key=lambda x: len(x))
    
    drop_words = []
    for idx,word in enumerate(words_to_add):
        if word in vocab_base:
            drop_words.append(idx)
    
    for idx in drop_words[::-1]: del words_to_add[idx]
    
    for word in words_to_add:
        
        split = tokenizer_base.tokenize(word if not word.startswith('Ġ') else ' '+word[1:])

        if len(split) == 1:
            continue
        
        if len(split) == 2: #pass
            vocab_to_merges[word] = [split[0],split[1]]
        
        if len(split) >= 3:
            new_split = checkList(split, vocab_to_merges)
            if len(new_split) == 2:
                vocab_to_merges[word] = [new_split[0],new_split[1]]
                continue
            
            new_word=new_split[0]
            for i in range(1,len(new_split)):
                left = new_word
                right = new_split[i]
                new_word += new_split[i]
                if new_word not in vocab_to_merges:
                    vocab_to_merges[new_word] = [left,right]
                
    
    idx = 0
    for key,val in vocab_to_merges.items():
        if key not in tokenizer_json["model"]["vocab"]:
            tokenizer_json["model"]["vocab"][key] = len(tokenizer_json["model"]["vocab"])
            tokenizer_json["model"]["merges"].append(val)

    dump_dir = f'./VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_{fname.split("/")[-1][:-4]}'
    tokenizer_base.save_pretrained(dump_dir)
    
    with open(dump_dir+'/tokenizer.json', 'w', encoding='utf-8') as f:
        json.dump(tokenizer_json, f)
    f.close()
    
    vocab_base_dump = json.load(open(dump_dir+'/vocab.json', 'r', encoding='utf-8'))
    idx = 0
    for key,val in vocab_to_merges.items():
        vocab_base_dump[key] = len(vocab_base_dump)
    
    with open(dump_dir+'/vocab.json', 'w', encoding='utf-8') as f:
        json.dump(vocab_base_dump, f)
    f.close()
    
    merge_base_dump = open(dump_dir+'/merges.txt', 'r', encoding='utf-8').read()
    for item in vocab_to_merges.values():
        merge_base_dump += '\n' + ' '.join(item)
    
    with open(dump_dir+'/merges.txt', 'w', encoding='utf-8') as f:
        f.write(merge_base_dump)
    f.close()
    
    tokenizer_added_vocab = AutoTokenizer.from_pretrained(dump_dir)
    tokenizer_added_vocab.save_pretrained(dump_dir)

In [ ]:
import pandas as pd
df = pd.read_csv('../../data/Legal_Val_OOV_Tokens.csv')

freq_ebm = df['Count'].to_list()
terms_EBM = df['Word'].to_list()
split_bart = df['Splits'].to_list()

sum_num = 0.
sum_den = 0.
for idx,term in enumerate(terms_EBM):
    sum_num += split_bart[idx]*freq_ebm[idx]
    sum_den += freq_ebm[idx]

old_score = sum_num/sum_den


import glob
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer

tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")

dict_scores = defaultdict(lambda : defaultdict(dict))
for fname in tqdm(sorted(glob.glob('./VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/*'), key=lambda x: float(x.split('_')[-2][:-1]))): 
    # print('***********Processing:',fname, fname.split('/')[-1].split('_')[1])
    try:
        domain_tok = AutoTokenizer.from_pretrained(fname)
        
        sum_num = 0.
        sum_den = 0.
        
        for idx,term in enumerate(terms_EBM):
            sum_num += len(domain_tok.tokenize(term))*freq_ebm[idx]
            sum_den += freq_ebm[idx]

        dict_scores[fname.split('/')[-1].split('_')[-2]] = [round(sum_num/sum_den,5),len(domain_tok)-len(tokenizer_base)]
    except Exception as e:
        print('Error:', e,fname)
        continue


with open(f'./CPT-Eval-SupremeCourtCases-FERTILITY','a') as f:
    f.write(f'\n-------------\nQwen2.5-7B: []\n--------------------\n')
    f.write('BASE_Tok: '+str(round(old_score,2))+'\n')
    
    for key,val in dict_scores.items():
        f.write(f'{key} : {val[0]} || {val[1]}\n')


In [1]:
import pandas as pd
df_train_oov = pd.read_csv('../../data/Legal_Train_OOV_Tokens.csv')

In [2]:
from transformers import AutoTokenizer

tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
tokenizer_added_vocab = AutoTokenizer.from_pretrained('./VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_/')

In [3]:
added_vocab = set(tokenizer_added_vocab.get_vocab().keys()) - set(tokenizer_base.get_vocab().keys())
print('Added Vocab Size:', len(added_vocab))

added_vocab_list = list(added_vocab)
added_vocab_list = sorted(added_vocab_list, key=lambda x: len(x))


Added Vocab Size: 10687


In [4]:
added_vocab_list

['CJ',
 'xv',
 'aJ',
 'uq',
 'Bh',
 'KU',
 'LJ',
 'OY',
 'JU',
 'CTE',
 'Sri',
 'Pvt',
 'ESI',
 'taf',
 'rao',
 'RIM',
 'jea',
 'crl',
 'aik',
 'WAL',
 'ĠJU',
 'JUD',
 'SEB',
 'mam',
 'heb',
 'adu',
 'aqf',
 'ĠSs',
 'akf',
 'noi',
 'YAS',
 'HAK',
 'BBS',
 'HAD',
 'eal',
 'LAL',
 'lak',
 'RIN',
 'NAD',
 'Bri',
 'hew',
 'HRA',
 'TAN',
 'Vij',
 'HAP',
 'KRI',
 'kia',
 'eek',
 'SCL',
 'mpt',
 'INN',
 'CRL',
 'HOL',
 'Raj',
 'GAT',
 'JAG',
 'AFT',
 'ĠOj',
 'BDS',
 'jot',
 'Rup',
 'MCO',
 'NAT',
 'pse',
 'ĠHU',
 'Rao',
 'kam',
 'IDC',
 'sue',
 'yee',
 'GAR',
 'ANJ',
 'FIC',
 'GOs',
 'wah',
 'eem',
 'AIK',
 'DCs',
 'ugn',
 'ESU',
 'VII',
 'lud',
 'MAD',
 'ALI',
 'Exh',
 'eev',
 'bri',
 'CUT',
 'whe',
 'YAN',
 'HIK',
 'hru',
 'EER',
 'tia',
 'KAM',
 'ERI',
 'PSC',
 'THI',
 'DIA',
 'sof',
 'AID',
 'ahs',
 'DIT',
 'ukh',
 'Lal',
 'DEE',
 'ĠKU',
 'waj',
 'bly',
 'CJI',
 'HHC',
 'DEY',
 'JEA',
 'NED',
 'Lok',
 'asw',
 'VRS',
 'YER',
 'DRA',
 'FSL',
 'PSU',
 'GDE',
 'Cer',
 'rue',
 'JEE',
 'ufr',
 

In [5]:
df_train_oov.describe(percentiles=[0.25,0.5,0.75,0.9,0.95,0.99])
df_train_oov = df_train_oov[df_train_oov['Count']>=13]
df_train_oov

,Word,Splits,Count
0,appellants,2,63684
1,impugned,3,31380
2,aforesaid,4,30407
3,assessee,2,29351
4,Ors,2,28183
...,...,...,...
33178,Harbansing,4,13
33179,Mukarji,3,13
33180,Maloth,2,13
33181,Kanel,2,13


In [6]:
import tqdm

potential_whole_words = []

def is_match(word, token):
    if token.startswith('Ġ'):
        token = token[1:]
    
    if word.strip() == token.strip():
        return True

    return False 

for word in tqdm.tqdm(added_vocab_list, desc='Finding Matches', total=len(added_vocab_list)):
    mask = df_train_oov['Word'].apply(lambda x: is_match(x, word))

    if not df_train_oov[mask].empty:
        potential_whole_words.append([word, df_train_oov[mask]['Count'].sum()])


Finding Matches: 100%|██████████| 10687/10687 [01:04<00:00, 166.73it/s]


In [7]:
potential_whole_words = sorted(potential_whole_words, key=lambda x: x[1], reverse=True)
potential_whole_words = [x[0].replace('Ġ','') for x in potential_whole_words]

potential_whole_words = list(set(potential_whole_words))
potential_whole_words = sorted(potential_whole_words, key=lambda x: len(x))
potential_whole_words

['KI',
 'CJ',
 'Sd',
 'Ws',
 'xv',
 'Ss',
 'Ld',
 'aJ',
 'yd',
 'Pw',
 'Dua',
 'Sri',
 'Pvt',
 'DPC',
 'ESI',
 'Jai',
 'PLP',
 'Jit',
 'Adi',
 'POs',
 'crl',
 'SHO',
 'SEB',
 'Mfg',
 'Mst',
 'Asa',
 'MRC',
 'HAD',
 'UDD',
 'XXV',
 'LAL',
 'HRA',
 'gur',
 'SRI',
 'CRL',
 'Nai',
 'EIA',
 'RAJ',
 'LRS',
 'Raj',
 'AFT',
 'UDC',
 'Rup',
 'RDS',
 'MRP',
 'LRs',
 'SGL',
 'SBI',
 'UCC',
 'EPF',
 'CEC',
 'BEG',
 'IBC',
 'Dey',
 'STs',
 'HSN',
 'HUF',
 'XIX',
 'Jee',
 'DoT',
 'Deo',
 'OBC',
 'RAO',
 'CVC',
 'BDA',
 'GOs',
 'HON',
 'Anr',
 'TPC',
 'nay',
 'VII',
 'ALI',
 'Exh',
 'HCL',
 'SCs',
 'ICI',
 'Exs',
 'MII',
 'LOK',
 'Etc',
 'MRF',
 'RDB',
 'DGP',
 'NTC',
 'BSF',
 'BOB',
 'PSC',
 'NOC',
 'Kha',
 'UAS',
 'SCB',
 'bly',
 'SIR',
 'CJI',
 'HHC',
 'EPC',
 'DEY',
 'Kgs',
 'Lok',
 'VRS',
 'TDS',
 'ATC',
 'Jus',
 'FSL',
 'PSU',
 'PVT',
 'Viz',
 'DRT',
 'LPG',
 'LAO',
 'WLR',
 'ous',
 'evi',
 'GOI',
 'FIR',
 'Wad',
 'Tis',
 'gms',
 'RPA',
 'Jat',
 'RIL',
 'STC',
 'Cus',
 'XXI',
 'UCO',
 'Lrs',
 '

In [8]:
len(potential_whole_words)

6187

In [9]:
with open('./VocabFiles/CPT-SupremeCourtCases/CPT-SupremeCourtCases-Qwen2.5_Vocab/Qwen2.5-7B-PotentialWholeWords.txt','w',encoding='utf-8') as f:
    for item in potential_whole_words:
        f.write(item.strip()+'\n')


In [ ]:
potential_whole_words = open('./VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-PotentialWholeWords.txt','r',encoding='utf-8').read().splitlines()
potential_whole_words = [x.strip() for x in potential_whole_words]
potential_whole_words = sorted(potential_whole_words, key=lambda x: len(x))

In [10]:
from transformers import AutoTokenizer
tokenizer_base = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
tokenizer_base.add_tokens(potential_whole_words)

tokenizer_base.save_pretrained("./VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_WholeWords/")

('./VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_WholeWords/tokenizer_config.json',
 './VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_WholeWords/special_tokens_map.json',
 './VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_WholeWords/vocab.json',
 './VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_WholeWords/merges.txt',
 './VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_WholeWords/added_tokens.json',
 './VocabFiles/CPT-SupremeCourtCases/Qwen2.5-7B-Tokenizer/SupremeCourtCases_MEDVOC-LLM_Qwen2.5_10.0_WholeWords/tokenizer.json')